# Gap-filling head time series using Pastas

*Raoul Collenteur, Hydroconsult, 2026*

This notebook shows how to use the Pastas gapfilling module to gapfill a head time series using a simulation from a Pastas model. Rather than 'simply' imputing the simulation values in the gaps, the simulation is corrected using the measurements surrounding the gap and the noise model (if available). Making use of a hydrological model (here, a Pastas model), ensure that we take the hydrology causing the head fluctuations into account. This is different from interpolation, which ignores the fact that it did or did not rain, for example. As usual, we start by importing the necessary Python packages.

The development of this module was sponsored by Waterboard Vallei & Veluwe.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

import pastas as ps

ps.show_versions()

## Example data

We look at the data from Rhode Island (USA), where the head measurements have an approximately daily frequency. We'll make some synthetic gaps that we can fill using a Pastas Model and the gapfill module. We only use every 14th measurement, and create a gap for the entire year with the lowest measured head (2007). 

In [ ]:
data = ps.load_dataset("collenteur_2019")
head = data["head"].squeeze().loc["2002":"2012"]

head_gap = head.iloc[::10].copy()
head_gap.loc["2007"] = pd.NA
head_gap = head_gap.dropna()
prec = data["rain"].asfreq("D").fillna(0) * 0.3048 * 1e3
evap = data["evap"].asfreq("D").interpolate() * 0.3048 * 1e3

## Make a Pastas Model

We'll make a simple Pastas Model based on precipitation and potential evaporation to simulate the head at a daily time step. The simulation time step from the Pastas Model is also the time step for the gapfilled time series later. 

In [ ]:
ml = ps.Model(head_gap)

sm = ps.RechargeModel(
    ml,
    prec=prec,
    evap=evap,
    rfunc=ps.Exponential(),
    name="rch",
    recharge=ps.rch.FlexModel(),
)

ml.solve(report=False)
ps.ArNoiseModel(ml)
ml.solve(initial=False, report=False)

We'll make a plot of all the data and the Pastas model that we generated so far. The figure below clearly shows how the simulation works quite well, but actually less good during the large gap with the minimum heads.

In [ ]:
ax = ml.plot(
    figsize=(10, 3),
)

head.plot(
    marker=".",
    color="gray",
    linestyle="None",
    label="head measured",
    markersize=3,
    zorder=1,
    ax=ax,
)

ax.grid()
ax.set_xlim("2003", "2013")
ax.legend()
ax.set_ylabel("Head (m)")

## Naive gapfilling

We'll start by filling the heads, without using the noisemodel. By default, the gapfill module will use the noisemodel if present, but we can manually turn this option off (`use_noisemodel=False`). The simulated head is used to gapfill the data, corrected by the last measurement before and the first measurement after the gap. In between, a linear interpolation is used for the correction.

By providing the `sig_level`, the lower and upper bounds for the (1-sig_level)*100%-prediction interval are computed. If use_noisemodels=False, the variance of the residuals will be used to compute the prediction interval, and will thus have a constant width.  

In [ ]:
head_filled = ps.gap_fill(
    head_gap.asfreq("D"),
    ml,
    use_noisemodel=False,
    sig_level=0.05,
)

In [ ]:
ax = ml.plot(
    figsize=(10, 3),
)

head.plot(
    marker=".",
    color="gray",
    linestyle="None",
    label="head measured",
    markersize=3,
    zorder=1,
    ax=ax,
)

ax.grid()
ax.set_xlim("2003", "2013")
ax.legend()
ax.set_ylabel("Head (m)")
head_filled["head_filled"].plot(
    label="head filled",
    markersize=3,
    zorder=2,
    ax=ax,
)
ax.fill_between(
    head_filled.index,
    head_filled["lower"],
    head_filled["upper"],
    color="lightblue",
    alpha=0.5,
)

ax.legend()

## Using the noisemodel

The methods applied above will work on any Pastas model to fill gaps in the data and obtain an uncertainty estimate. If a noisemodel is used, however, the noise model can be used to correct the simulated heads based on the last residual before and the first residual after the gap. The noisemodel quantifies how the residual behaves over time, and we can use that relationship to correct the simulation based on the expected (but unknown) residuals in the gap, conditioned on the known residuals before and after. Moreover, the prediction interval can also use this information, causing the intervals to widen further away from the last measurement.



In [ ]:
ml.plots.diagnostics();

In [ ]:
head_filled_noise = ps.gap_fill(
    head_gap.asfreq("D"),
    ml,
    use_noisemodel=True,
    sig_level=0.05,
)


In [ ]:
ax = ml.plot(
    figsize=(10, 3),
)

head.plot(
    marker=".",
    color="gray",
    linestyle="None",
    label="head measured",
    markersize=3,
    zorder=1,
    ax=ax,
)

ax.grid()
ax.set_xlim("2003", "2013")
ax.legend()
ax.set_ylabel("Head (m)")
head_filled_noise["head_filled"].plot(
    label="head filled",
    markersize=3,
    zorder=2,
    ax=ax,
)
ax.fill_between(
    head_filled_noise.index,
    head_filled_noise["lower"],
    head_filled_noise["upper"],
    color="lightgray",
    alpha=0.8,
    zorder=0,
)

ax.legend()


## Performance of the gapfilled heads

In [ ]:
rsq_noise = ps.stats.metrics.rsq(
    head.loc[:"2012-12-26"], head_filled_noise["head_filled"]
)

rsq = ps.stats.metrics.rsq(head.loc[:"2012-12-26"], head_filled["head_filled"])

print("RSQ with noise model:", f"{rsq_noise:.3f}")
print("RSQ without noise model:", f"{rsq:.3f}")

## PICP